In [ ]:
import re
import cv2
import os
from os import listdir
from scipy import interpolate
import numpy as np
import pandas as pd
import torch.nn.functional as F


def is_float(s):
    try:
        float(s)
        return True
    except Exception:
        return False

def jcamp_read(filehandle):
    """
    Robust JCAMP-DX reader for NIST and similar files.
    Handles XYDATA, compressed (X++(Y..Y)), and uncompressed data.
    """
    jcamp_dict = {}
    x = []
    y = []
    xfactor = 1.0
    yfactor = 1.0
    in_data = False
    data_format = None
    deltax = None
    last_x_value = None # New state variable for compressed data

    for line in filehandle:
        if hasattr(line, 'decode'):
            line = line.decode('utf-8', 'ignore')
        line = line.strip()
        if not line or line.startswith('$$'):
            continue

        # Header lines
        if line.startswith('##'):
            key, _, value = line[2:].partition('=')
            key = key.strip().lower()
            value = value.strip()
            jcamp_dict[key] = value
            
            if key == 'xfactor':
                try: xfactor = float(value)
                except Exception: xfactor = 1.0
            if key == 'yfactor':
                try: yfactor = float(value)
                except Exception: yfactor = 1.0
            if key == 'deltax':
                try: deltax = float(value)
                except Exception: deltax = None
            if key in ('xydata', 'peak table', 'xypoints'):
                in_data = True
                data_format = value.lower()
                continue
            if key == 'end':
                in_data = False
                continue
            continue

        # Data lines
        if in_data:
            if line.startswith('##') or line.startswith('$$'):
                continue
            
            if data_format.startswith('(x++(y..y))'):
                
                # Use simple Python split on whitespace
                tokens = [t for t in line.split() if t]
                
                if not tokens:
                    continue
                try:
                    
                    if not x: # First line of the data block
                        # Expects X0 Y1 Y2 Y3... format
                        if len(tokens) < 2: continue
                        
                        x0 = float(tokens[0]) * xfactor
                        yvals = [float(val) * yfactor for val in tokens[1:] if is_float(val)]
                        
                        # Set initial X value and point
                        x.append(x0)
                        y.append(yvals[0])
                        yvals.pop(0)
                        
                        last_x_value = x0
                        
                    else: # Subsequent lines only contain Y values
                        # Assume all tokens are Y values; X continues from last_x_value
                        yvals = [float(val) * yfactor for val in tokens if is_float(val)]
                    
                    if deltax is None:
                        # Fallback for DELTAX, should use 5.0 for IR if header is missing
                        deltax = 5.0
                    
                    # Append all points using deltax
                    for yval in yvals:
                        last_x_value += deltax
                        x.append(last_x_value)
                        y.append(yval)
                        
                except Exception as e:
                    # Catch and print the error for debugging.
                    # print(f"DEBUG: Parsing failure on line: {line}. Error: {e}") 
                    continue 
            
            # Note: Other data formats (e.g., XYPOINTS) would need their own logic here

    if len(x) == 0 or len(y) == 0:
        raise ValueError("No data found in JCAMP file or unable to parse x/y arrays.")

    jcamp_dict['x'] = np.array(x)
    jcamp_dict['y'] = np.array(y)
    return jcamp_dict

def jcamp_readfile(filename):
    with open(filename, 'rb') as filehandle:
        return jcamp_read(filehandle)

In [ ]:
def convert_x(x_in, unit_from, unit_to):
    """Convert between micrometer and wavenumber."""
    if unit_to == 'micrometers' and unit_from == 'MICROMETERS':
        x_out = x_in
        return x_out
    elif unit_to == 'cm-1' and unit_from in ['1/CM', 'cm-1', '1/cm', 'Wavenumbers (cm-1)']:
        x_out = x_in
        return x_out
    elif unit_to == 'micrometers' and unit_from in ['1/CM', 'cm-1', '1/cm', 'Wavenumbers (cm-1)']:
        x_out = np.array([10 ** 4 / i for i in x_in])
        return x_out
    elif unit_to == 'cm-1' and unit_from == 'MICROMETERS':
        x_out = np.array([10 ** 4 / i for i in x_in])
        return x_out

def convert_y(y_in, unit_from, unit_to):
    """Convert between absorbance and trasmittance."""
    if unit_to == 'transmittance' and unit_from in ['% Transmission', 'TRANSMITTANCE', 'Transmittance']:
        y_out = y_in
        return y_out
    elif unit_to == 'absorbance' and unit_from == 'ABSORBANCE':
        y_out = y_in
        return y_out
    elif unit_to == 'transmittance' and unit_from == 'ABSORBANCE':
        y_out = np.array([1 / 10 ** j for j in y_in])
        return y_out
    elif unit_to == 'absorbance' and unit_from in ['% Transmission', 'TRANSMITTANCE', 'Transmittance']:
        y_out = np.array([np.log10(1 / j) for j in y_in])
        return y_out

def get_unique(x_in, y_in):
    """Removes duplicates in x and takes smallest y value for each x value."""
    x_out = sorted(list(set(x_in)), reverse=True)
    y_out = []
    for i in x_out:
        y_temp = []
        for ii, j in zip(x_in, y_in):
            if i == ii:
                y_temp.append(j)
        y_out.append(min(y_temp))
    return x_out, y_out

def convert_transmittance_to_absorbance(T_data: np.ndarray, base=10) -> np.ndarray:
    """
    Converts Transmittance (T) values to Absorbance (A) values.
    
    A = -log10(T)
    
    Args:
        T_data: NumPy array of Transmittance values (T, between 0 and 1).
        base: Logarithm base (10 for IR spectroscopy).
        
    Returns:
        NumPy array of Absorbance values (A).
    """
    # Protect against T=0, which causes log10(0) = -inf
    # Use a small epsilon (1e-6) if T is close to zero.
    epsilon = 1e-6
    T_clamped = np.clip(T_data, epsilon, 1.0)
    
    # Calculate Absorbance (A = -log10(T))
    A_data = -np.log10(T_clamped)
    
    return A_data

def get_contours(image):
    """Returns normalized coordinates of a spectrum."""
    # image = cv2.imread(sdbs_png_path + '/' + file, 0)
    ret, thresh = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY_INV)
    # Define kernel length.
    kernel_length = np.array(image).shape[1] // 80
    # Verticle kernel of (1 * kernel_length) used to detect verticle lines in the image.
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_length))
    # Horizontal kernel of (kernel_length * 1) used to detect horizontal lines in the image.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_length, 1))
    # Kernel of (3 X 3) ones.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    # Morphological operation to detect vertical lines from an image.
    vertical_temp = cv2.erode(thresh, vertical_kernel, iterations=3)
    verticle_lines = cv2.dilate(vertical_temp, vertical_kernel, iterations=3)
    # Morphological operation to detect horizontal lines from an image.
    horizontal_temp = cv2.erode(thresh, horizontal_kernel, iterations=3)
    horizontal_lines = cv2.dilate(horizontal_temp, horizontal_kernel, iterations=3)
    # Add two images with specific weight parameters to get a third summation image.
    image = cv2.addWeighted(verticle_lines, 0.5, horizontal_lines, 0.5, 0.0)
    image = cv2.erode(~image, kernel, iterations=2)
    ret, thresh = cv2.threshold(image, 127,255, cv2.THRESH_BINARY)
    # Find contours in the image
    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    return contours

In [ ]:
sdbs_png_path = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data\gif"
sdbs_other_path = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data\other"
out_csv_path = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data"
os.makedirs(out_csv_path, exist_ok=True)

combined_csv_path = os.path.join(out_csv_path, "sdbs_dataset.csv")
skipped_csv_path = os.path.join(out_csv_path, "sdbs_skipped.csv")


def load_inchi(sdbs_id):
    other_file = os.path.join(sdbs_other_path, f"{sdbs_id}_other.txt")
    if not os.path.exists(other_file):
        return None

    with open(other_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            match = re.match(r'InChI:\s*(.*)', line.strip())
            if match:
                value = match.group(1).strip()
                return value if value else None
    return None


def get_plot_region(gray):
    contours = get_contours(gray)
    best = None
    best_area = 0

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        if w >= 350 and h >= 150 and area > best_area:
            best = (x, y, w, h)
            best_area = area

    if best is None:
        return gray

    x, y, w, h = best
    x0 = max(x - 2, 0)
    y0 = max(y - 2, 0)
    x1 = min(x + w + 2, gray.shape[1])
    y1 = min(y + h + 2, gray.shape[0])
    return gray[y0:y1, x0:x1]


def extract_transmittance_curve(plot_img):
    h, w = plot_img.shape
    if h < 80 or w < 150:
        return None, None

    # Dark spectrum line on light background.
    binary = plot_img < 110

    x_vals = []
    y_vals = []

    row_low = int(0.05 * h)
    row_high = int(0.95 * h)

    for col in range(w):
        rows = np.where(binary[:, col])[0]
        if rows.size == 0:
            continue

        rows = rows[(rows >= row_low) & (rows <= row_high)]
        if rows.size == 0:
            continue

        # top-most dark pixel approximates spectrum trace position.
        row = int(rows.min())

        wn = 4000.0 - (col / max(w - 1, 1)) * 3600.0
        trans = (h - 1 - row) / max(h - 1, 1)

        x_vals.append(wn)
        y_vals.append(trans)

    if len(x_vals) < 50:
        return None, None

    x_vals, y_vals = get_unique(x_vals, y_vals)
    return np.array(x_vals), np.array(y_vals)


skipped = []
missing_meta = []
processed = 0
combined_rows = []

x_grid = np.linspace(4000, 400, 1648)

print('Start SDBS preprocessing')
for file in listdir(sdbs_png_path):
    lower = file.lower()
    if not lower.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
        continue

    file_path = os.path.join(sdbs_png_path, file)
    sdbs_id = os.path.splitext(file)[0]

    try:
        gray = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
        if gray is None:
            skipped.append((sdbs_id, 'image_read_failed'))
            continue

        inchi = load_inchi(sdbs_id)
        if not inchi:
            inchi = ''
            missing_meta.append((sdbs_id, 'missing_inchi'))

        plot_img = get_plot_region(gray)
        x_vals, y_vals = extract_transmittance_curve(plot_img)
        if x_vals is None:
            skipped.append((sdbs_id, 'curve_extraction_failed'))
            continue

        interp_fn = interpolate.interp1d(
            x_vals,
            y_vals,
            kind='cubic',
            bounds_error=False,
            fill_value='interpolate'
        )
        y_interp = interp_fn(x_grid)
        y_interp = np.clip(y_interp, 0.0, 1.0)

        y_abs = convert_transmittance_to_absorbance(y_interp)

        row = [sdbs_id] + list(y_abs)
        combined_rows.append(row)
        processed += 1

    except Exception as e:
        skipped.append((sdbs_id, f'process_error: {e}'))
        continue

feature_cols = [f'{i}' for i in range(len(x_grid))]
combined_df = pd.DataFrame(combined_rows, columns=['sdbs_id'] + feature_cols)
combined_df.to_csv(combined_csv_path, index=False)

skipped_df = pd.DataFrame(skipped, columns=['sdbs_id', 'reason'])
skipped_df.to_csv(skipped_csv_path, index=False)

print(f"Processed: {processed}")
print(f"Skipped: {len(skipped)}")
print(f"Missing metadata (processed anyway): {len(missing_meta)}")
print(f"Combined CSV saved: {combined_csv_path}")
print(f"Skipped report saved: {skipped_csv_path}")
for sid, reason in skipped[:20]:
    print(f" - {sid}: {reason}")
if missing_meta:
    print('Missing metadata sample:')
    for sid, reason in missing_meta[:20]:
        print(f" - {sid}: {reason}")


Start SDBS preprocessing
Processed: 4
Skipped: 0
Missing metadata (processed anyway): 4
Combined CSV saved: D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_dataset\sdbs_dataset1.csv
Skipped report saved: D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_dataset\sdbs_skipped.csv
Missing metadata sample:
 - 1: missing_inchi
 - 3: missing_inchi
 - 5: missing_inchi
 - 6: missing_inchi
